# Runs, events, and sessions

Goal: contrast `start` vs `run`, read `EventBatch` values, attach a `PythonObserver`, and inspect a session lane.

Trust: T2 callback. Network: none. Dropping a `Run` handle detaches observation; it does not cancel.


In [ ]:
import asyncio
from typing import Any

import finstack_ai


async def model(
    context: finstack_ai.CallbackContext, request: dict[str, Any]
) -> dict[str, Any]:
    del context, request
    return {"text": "ready", "completion_id": "notebook-run-1"}


model_port = finstack_ai.PythonModel(
    model,
    component="notebook.model.runs",
    provider="notebook",
    model="notebook-model",
)
seen: list[dict[str, Any]] = []


async def observe(batch: list[dict[str, Any]]) -> None:
    seen.extend(batch)


agent = await finstack_ai.Agent.from_python(
    model_port,
    observers=[
        finstack_ai.PythonObserver(
            observe,
            component="notebook.observer.runs",
            payload_mode="redacted",
        )
    ],
)
direct = await agent.run("hello")
print(direct.text)

In [ ]:
async def _collect(handle: finstack_ai.Run) -> list[finstack_ai.EventBatch]:
    return [batch async for batch in handle.events()]


run = agent.start("hello again")
result, batches = await asyncio.gather(
    run.result(),
    _collect(run),
)
print(result.text)
print(len(batches), batches[-1].events()[-1].kind)
print(any(event.get("kind") == "run_completed" for event in seen))

In [ ]:
session = await agent.create_session()
lane = await session.lane("main")
info = await lane.inspect()
print(session.session_id)
print(info["name"], info["history_len"])